In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Handles BOTH cases: if /content/3DPinball-DrosophilaCadet already exists (runtime survived),
# this does nothing and just prints 'already present'. If it's a fresh VM, this re-extracts the
# original project zip from Drive and runs the full build/dependency setup script.
import os
if os.path.isdir('/content/3DPinball-DrosophilaCadet'):
    print('already present - skipping fresh setup')
else:
    print('fresh VM detected - re-extracting project and running full setup')
    os.system('cp /content/drive/MyDrive/pinball_checkpoints/3DPinball-DrosophilaCadet.zip /content/ && cd /content && unzip -q 3DPinball-DrosophilaCadet.zip')
    os.system('cd /content/3DPinball-DrosophilaCadet && bash colab_setup.sh')

In [ ]:
# Always applies the RNG-reproducibility fix on top, whether this VM is fresh or survived -
# the base project zip predates the fix either way.
!cp /content/drive/MyDrive/pinball_checkpoints/rng_fix.zip /content/ && cd /content/3DPinball-DrosophilaCadet && unzip -o /content/rng_fix.zip

In [ ]:
# Rebuilds (harmless no-op if colab_setup.sh already built it fresh just above).
!cd /content/3DPinball-DrosophilaCadet && cmake -S vendor/SpaceCadetPinball -B build-native -DCMAKE_BUILD_TYPE=Debug && cmake --build build-native --parallel

In [ ]:
!ls -la /content/3DPinball-DrosophilaCadet/vendor/SpaceCadetPinball/bin/

In [ ]:
!pkill -9 -f watchdog.sh; pkill -9 -f train_pinball_circuit_cem; pkill -9 -f SpaceCadetPinball; sleep 2; echo done

In [ ]:
%%writefile /content/watchdog_v2.sh
#!/bin/bash
cd /content/3DPinball-DrosophilaCadet
CKPT=/content/drive/MyDrive/pinball_checkpoints/pinball_circuit_cem_595n_v2.pt
TARGET_GEN=400
while true; do
  if [ -f "$CKPT" ]; then
    GEN=$(python3 -c "import torch; print(torch.load('$CKPT', weights_only=False)['generation'])" 2>/dev/null)
    RESUME_ARG="--resume $CKPT"
  else
    GEN=0
    RESUME_ARG=""
  fi
  echo "WATCHDOG_V2: $(date) current saved generation: $GEN" >> /content/train_v2.log
  if [ -n "$GEN" ] && [ "$GEN" -ge "$TARGET_GEN" ]; then
    echo "WATCHDOG_V2: reached generation $GEN, stopping" >> /content/train_v2.log
    break
  fi
  python agents/train_pinball_circuit_cem.py --binary vendor/SpaceCadetPinball/bin/SpaceCadetPinball --connectome web/connectome.json --readout-dim 32 --workers 6 --population 16 --elites 4 --generations $TARGET_GEN --courses-per-candidate 2 --validation-episodes 4 --frame-skip 4 --max-steps 800 $RESUME_ARG --out "$CKPT" --save-every 5 < /dev/null >> /content/train_v2.log 2>&1
  echo "WATCHDOG_V2: $(date) training process exited with code $?, restarting in 5s" >> /content/train_v2.log
  sleep 5
done

In [ ]:
import subprocess
with open('/content/watchdog_v2_stdout.log', 'w') as log_f:
    subprocess.Popen(['setsid', 'bash', '/content/watchdog_v2.sh'], stdin=subprocess.DEVNULL,
                      stdout=log_f, stderr=subprocess.STDOUT, start_new_session=True)
print('launched')

In [ ]:
!grep -a champion /content/train_v2.log | tail -8; echo ---; ps aux | grep -c SpaceCadetPinball